# GPU Hypertrain Notebook - Units SAEs layers 24/28

This notebook is a **separate tuning attempt** for the units result. It does not overwrite the original SAE checkpoints.

Goal:

- train units-specialized SAEs for layers 24 and 28 only;
- keep the same latent dimension (`8192`) so they can be mixed with the old checkpoints;
- save tuned checkpoints to Drive under a new folder;
- assemble a mixed checkpoint directory: old SAEs for all other layers, tuned SAEs for 24/28;
- rerun only the units experiments affected by SAE quality, with distinct output names.

This is intentionally small. It tests whether better units-specific SAEs improve sparse feature interventions without starting a giant hyperparameter sweep.


## Step 1 - Mount Drive and fetch code

GitHub is preferred for code. Drive `project.zip` remains a fallback. SAE checkpoints and tuned outputs persist in Drive.


In [ ]:
# Step 1: Mount Google Drive and fetch project code
# GitHub is the primary code source. Drive project.zip remains a backup.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

repo_url = "https://github.com/evey-dev/test_run.git"
repo_dir = "/content/test_run"

def run_cmd(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

github_ok = False
try:
    clone_dir = repo_dir
    if os.path.isdir(os.path.join(clone_dir, ".git")):
        run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
    else:
        if os.path.exists(clone_dir) and os.listdir(clone_dir):
            clone_dir = "/content/test_run_github"
        if os.path.isdir(os.path.join(clone_dir, ".git")):
            run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
        elif os.path.exists(clone_dir) and os.listdir(clone_dir):
            raise RuntimeError(f"{clone_dir} exists but is not a git repo")
        else:
            run_cmd(["git", "clone", "--depth", "1", repo_url, clone_dir])
    os.chdir(clone_dir)
    github_ok = True
    print(f"Using GitHub checkout: {os.getcwd()}")
except Exception as exc:
    print("GitHub checkout failed; falling back to Drive project.zip.")
    print(repr(exc))

if not github_ok:
    zip_path = "/content/drive/MyDrive/mphil-project/project.zip"
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Could not find {zip_path}. Fix GitHub access or upload project.zip.")
    print(f"Extracting backup zip {zip_path}...")
    run_cmd(["unzip", "-q", "-o", zip_path, "-d", "/content/"])
    for candidate in ["/content/test_run", "/content/mphil_project/test_run", "/content"]:
        if os.path.isdir(os.path.join(candidate, "src")) and os.path.isdir(os.path.join(candidate, "configs")):
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError("Could not find extracted project root containing src/ and configs/.")
    print(f"Using Drive backup project: {os.getcwd()}")

print(f"Current working directory: {os.getcwd()}")
!ls -l


## Step 2 - Install dependencies


In [ ]:
# Step 2: Install dependencies
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .


## Step 3 - Generate units data


In [ ]:
# Step 3: Generate datasets
!python data/generate_datasets.py
print("
Dataset files:")
!ls -lh data/units_data.csv


## Step 4 - Restore original SAE checkpoints from Drive

These stay as the baseline checkpoint set. Later we build a mixed directory by copying these and overwriting only layers 24/28 with tuned checkpoints.


In [ ]:
# Restore original SAE checkpoints from Drive
import os, shutil, glob

drive_sae = '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints'
local_sae = '/content/test_run/mechanistic_data/sae_checkpoints'
os.makedirs(local_sae, exist_ok=True)

if os.path.isdir(drive_sae) and glob.glob(f'{drive_sae}/sae_layer*.pt'):
    for f in os.listdir(drive_sae):
        shutil.copy2(f'{drive_sae}/{f}', local_sae)
    print("Restored original SAE checkpoints from Drive:")
    !ls -lh {local_sae}
else:
    raise FileNotFoundError(f"No original SAE checkpoints found in Drive at {drive_sae}")


## Step 5 - Capture units-only activations for layers 24 and 28

This creates a separate activation bundle, `mechanistic_data_units_hyper`, so the original pooled activations are left untouched.


In [ ]:
# Capture units-only MLP activations for tuned SAE training
!python src/capture_activations.py \
  --output-dir mechanistic_data_units_hyper \
  --behaviours units \
  --layers 24 28 \
  --seed 787

print("\nUnits-only activation bundle:")
!ls -lh mechanistic_data_units_hyper


## Step 6 - Train tuned units SAEs for layers 24 and 28

Config: `configs/sae_units_l24_l28_hyper_config.yaml`

Differences from original:

- units-only activation data;
- layers 24 and 28 only;
- same latent dim (`8192`) for compatibility with old checkpoints;
- weaker sparsity (`l1_lambda=0.0003` instead of `0.001`);
- more epochs (`75` instead of `50`).

Tuned checkpoints are backed up to:

`/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_units_l24_l28_hyper`


In [ ]:
# Train tuned units SAEs for layers 24 and 28
# Write the config in the runtime too, so this notebook is robust even before a GitHub push.
from pathlib import Path

Path('configs/sae_units_l24_l28_hyper_config.yaml').write_text('output_dir: "mechanistic_data/sae_checkpoints_units_l24_l28_hyper"\ndata_dir: "mechanistic_data_units_hyper"\nlayers:\n  - 24\n  - 28\nhidden_size: 2560\nlatent_dim: 8192\nbatch_size: 64\nepochs: 75\nlr: 0.001\nl1_lambda: 0.0003\nseed: 787\ndevice: "auto"\n')
print('Wrote configs/sae_units_l24_l28_hyper_config.yaml')

!python src/train.py \
  --config configs/sae_units_l24_l28_hyper_config.yaml \
  --drive-dir /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_units_l24_l28_hyper

print("\nTuned local checkpoints:")
!ls -lh mechanistic_data/sae_checkpoints_units_l24_l28_hyper
print("\nTuned Drive checkpoints:")
!ls -lh /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_units_l24_l28_hyper


## Step 7 - Build mixed checkpoint directory

The normal scripts load one checkpoint directory. This cell creates a mixed directory:

- old checkpoints for layers 4,8,12,16,20;
- tuned units checkpoints for layers 24,28.

It also writes a runtime SAE config pointing at the mixed directory.


In [ ]:
# Build mixed checkpoint directory: old layers + tuned 24/28
import os, shutil, glob

old_dir = '/content/test_run/mechanistic_data/sae_checkpoints'
tuned_dir = '/content/test_run/mechanistic_data/sae_checkpoints_units_l24_l28_hyper'
drive_tuned_dir = '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_units_l24_l28_hyper'
mixed_dir = '/content/test_run/mechanistic_data/sae_checkpoints_units_hyper_mixed'

os.makedirs(mixed_dir, exist_ok=True)

# Start with the original checkpoint set.
for src in glob.glob(f'{old_dir}/*'):
    if os.path.isfile(src):
        shutil.copy2(src, mixed_dir)

# Prefer local tuned checkpoints; fall back to Drive if the runtime restarted.
source_tuned = tuned_dir if glob.glob(f'{tuned_dir}/sae_layer*.pt') else drive_tuned_dir
for layer in [24, 28]:
    for name in [f'sae_layer{layer}.pt', f'sae_layer{layer}_metadata.json', f'latents_layer{layer}.npy']:
        src = os.path.join(source_tuned, name)
        if not os.path.exists(src):
            raise FileNotFoundError(f'Missing tuned layer artifact: {src}')
        shutil.copy2(src, os.path.join(mixed_dir, name))

runtime_cfg = '
'.join([
    'output_dir: "mechanistic_data/sae_checkpoints_units_hyper_mixed"',
    'layers:',
    '  - 4',
    '  - 8',
    '  - 12',
    '  - 16',
    '  - 20',
    '  - 24',
    '  - 28',
    'hidden_size: 2560',
    'latent_dim: 8192',
    'batch_size: 64',
    'epochs: 75',
    'lr: 0.001',
    'l1_lambda: 0.0003',
    'seed: 787',
    'device: "auto"',
    '',
])
with open('/content/test_run/configs/sae_units_hyper_mixed_runtime.yaml', 'w') as fh:
    fh.write(runtime_cfg)

print("Mixed checkpoint directory:")
!ls -lh {mixed_dir}
print("
Runtime config:")
!cat configs/sae_units_hyper_mixed_runtime.yaml


## Step 8 - Rebuild units contrast graphs with mixed checkpoints

These graph outputs are named `units_hyper_*` so they do not overwrite the original units graphs.


In [ ]:
# Hyper graph 1: force -> newtons, contrast joules
!python src/attribution_graph.py \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --target "newtons" \
  --contrast-target "joules" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_hyper_mixed_runtime.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/units_hyper_force_graph.json \
  --output-html outputs/units_hyper_force_graph.html \
  --output-mermaid outputs/units_hyper_force_graph.md


In [ ]:
# Hyper graph 2: energy -> joules, contrast newtons
!python src/attribution_graph.py \
  --prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --target "joules" \
  --contrast-target "newtons" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_hyper_mixed_runtime.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/units_hyper_energy_graph.json \
  --output-html outputs/units_hyper_energy_graph.html \
  --output-mermaid outputs/units_hyper_energy_graph.md


## Step 9 - Rerun affected units interventions with mixed checkpoints

These are the cells most affected by SAE quality:

- full latent swap, because the SAE latent code for layers 24/28 changes;
- sparse feature swap, because the graph features are recomputed under the mixed SAE set;
- graph-positive feature inhibition, for the same reason.

Component knockouts are not repeated here because they do not depend meaningfully on SAE quality.


In [ ]:
# Hyper full latent swap: ENERGY -> FORCE
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_hyper_mixed_runtime.yaml \
  --target-token "newtons, joules, volts" \
  --positions all \
  --layer-scan \
  --output outputs/units_hyper_swap_minpair_energy_to_force.json


In [ ]:
# Hyper full latent swap: FORCE -> ENERGY
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_hyper_mixed_runtime.yaml \
  --target-token "newtons, joules, volts" \
  --positions all \
  --layer-scan \
  --output outputs/units_hyper_swap_minpair_force_to_energy.json


In [ ]:
# Hyper sparse swap: ENERGY graph-positive features -> FORCE run
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_hyper_mixed_runtime.yaml \
  --graph-json outputs/units_hyper_energy_graph.json \
  --graph-feature-sign positive \
  --target-token "newtons, joules, volts" \
  --positions all \
  --output outputs/units_hyper_swap_sparse_energy_to_force.json


In [ ]:
# Hyper sparse swap: FORCE graph-positive features -> ENERGY run
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_hyper_mixed_runtime.yaml \
  --graph-json outputs/units_hyper_force_graph.json \
  --graph-feature-sign positive \
  --target-token "newtons, joules, volts" \
  --positions all \
  --output outputs/units_hyper_swap_sparse_force_to_energy.json


In [ ]:
# Hyper sparse inhibition: remove force-graph positive features from FORCE run
!python src/intervention.py \
  --mode inhibit \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --target-token "newtons, joules, volts" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_hyper_mixed_runtime.yaml \
  --graph-json outputs/units_hyper_force_graph.json \
  --graph-feature-sign positive \
  --positions last \
  --output outputs/units_hyper_inhibit_force.json


## Step 10 - Persist hypertrain outputs to Drive


In [ ]:
# Copy hypertrain graphs/results to Drive
import os, shutil, glob

drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/test_run/outputs/units_hyper_*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')
